# Evaluating Prompt Engineering Methods On ABSA Dev

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_path = "drive/MyDrive/FYP/ALLAM-7B"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()



In [ ]:
def generate_chat(model, tokenizer, user_content: str, max_new_tokens: int):
    messages = [{"role": "user", "content": user_content}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,      # deterministic
            temperature=None,     # ignored when do_sample=False
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )
    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [ ]:
import json

In [ ]:
def extract_first_json(text:str):
    #this function aims to find {...} from the raw generated text
    if text is None:
        return None
    text = text.strip()
    return json.loads(text)

##### Changed the Extract First Json because it couldnt track the {...} block because it was nested in here 

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
        return rows

# Evaluating Prompt Engineering Methods On ABSA Dev

In [ ]:
absa_train_path = "/content/drive/MyDrive/FYP/absa/absa_train_big.jsonl"
absa_dev_path = "/content/drive/MyDrive/FYP/absa/absa_dev.jsonl"
absa_test_path = "/content/drive/MyDrive/FYP/absa/absa_test.jsonl"

absa_train = load_jsonl(absa_train_path)
absa_dev = load_jsonl(absa_dev_path)
absa_test = load_jsonl(absa_test_path)

In [ ]:
def get_allowed_categories(rows):
    allowed_categories = set()
    for r in rows:
        for op in r["labels"]:
            allowed_categories.add(op["category"])
    return allowed_categories

ALLOWED_CATEGORIES = get_allowed_categories(absa_train)

In [ ]:
def pick_absa_shots(rows, shots):
    if shots!=3 and shots!=5:
        raise ValueError("shots must be 3 or 5")
    picked_shots = []
    for r in rows:
        picked_shots.append(r)
        if len(picked_shots)==shots:
            break
    return picked_shots

absa_three_shots = pick_absa_shots(absa_train, 3)
absa_five_shots = pick_absa_shots(absa_train, 5)

In [ ]:
def build_absa_prompt(text, allowed_categories, picked_shots=None):
    cats_str = ", ".join(allowed_categories)

    header = (
        "أنت نظام لاستخراج مشاعر الجوانب (Aspect-Based Sentiment) من مراجعات الفنادق باللغة العربية.\n\n"
        "المهمة:\n"
        "استخرج جميع الآراء (opinions) الموجودة في النص.\n"
        "كل رأي يجب أن يكون زوجاً من:\n"
        "- category: جانب محدد بصيغة مثل HOTEL#CLEANLINESS\n"
        "- polarity: واحدة من {positive, negative, neutral, conflict}\n\n"
        "قيود مهمة جداً:\n"
        f"- استخدم فقط الفئات التالية (categories): {cats_str}\n"
        "صيغة الإخراج بالضبط:\n"
        '{"labels":[{"category":"...","polarity":"..."}, ...]}\n'
        "لا تكتب ولا تكتب اي شيء آخر '''json فقط اخرج بصياغ المكتوب اعلى\n"
        "- إذا لم تجد أي آراء، أرجع: {\"labels\":[]}\n"
        "لا تخمّن. لا تضف أي رأي إلا إذا كان واضحًا ومذكورًا في النص."
        "أخرج بحد أقصى 8 آراء. إذا زادت، اختر الأوضح فقط."
    )

    examples = ""
    if picked_shots:
        lines = ["أمثلة (Examples):"]
        for shot in picked_shots:
            lines.append(f'النص: {shot["text"]}')
            lines.append("الإجابة: " + json.dumps({"labels": shot["labels"]}, ensure_ascii=False))
            lines.append("")
        examples = "\n".join(lines).strip() + "\n\n"

    return f"{header}\n{examples}النص:\n{text}\n"

In [ ]:
def format_absa_prediction(obj, allowed_categories):
    extracted = extract_first_json(raw)